<a href="https://colab.research.google.com/github/Moquiuti/LangChainePython/blob/main/LangGraph_e_cadeias.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install -q langchain langgraph langchain-google-genai python-dotenv

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 67.6/67.6 kB 1.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 515.1/515.1 kB 12.5 MB/s eta 0:00:00


In [2]:
from google.colab import userdata
import os

api_key = userdata.get("python-ai-integrada")
os.environ["GOOGLE_API_KEY"] = api_key

print("Chave carregada com sucesso!" if api_key else "Chave não encontrada.")

Chave carregada com sucesso!


In [3]:
import asyncio
from typing import TypedDict, Literal

from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

from langgraph.graph import StateGraph, START, END

In [4]:
modelo = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash",
    temperature=0.5
)

Estado do grafo

In [5]:
class Estado(TypedDict):
    query: str
    destino: str
    resposta: str

Prompt do roteador

In [6]:
prompt_roteador = ChatPromptTemplate.from_messages([
    ("system", """
Você é um roteador de consultas de viagem.
Sua tarefa é analisar a consulta do usuário e responder somente com uma destas palavras:
- praia
- montanha

Não explique. Não escreva frases. Responda apenas com uma palavra.
"""),
    ("human", "{query}")
])

cadeia_roteador = prompt_roteador | modelo | StrOutputParser()

Prompt do assistente de praia

In [7]:
prompt_praia = ChatPromptTemplate.from_messages([
    ("system", """
Você é um consultor especialista em viagens para praia.
Dê sugestões objetivas, práticas e agradáveis para o usuário.
"""),
    ("human", "{query}")
])

cadeia_praia = prompt_praia | modelo | StrOutputParser()

Prompt do assistente de montanha

In [8]:
prompt_montanha = ChatPromptTemplate.from_messages([
    ("system", """
Você é um consultor especialista em viagens para montanha.
Dê sugestões objetivas, práticas e agradáveis para o usuário.
"""),
    ("human", "{query}")
])

cadeia_montanha = prompt_montanha | modelo | StrOutputParser()

Nó roteador assíncrono

In [9]:
async def no_roteador(estado: Estado) -> Estado:
    destino = await cadeia_roteador.ainvoke({"query": estado["query"]})
    destino = destino.strip().lower()

    if "praia" in destino:
        destino = "praia"
    else:
        destino = "montanha"

    return {
        **estado,
        "destino": destino
    }

Nó de praia

In [10]:
async def no_praia(estado: Estado) -> Estado:
    resposta = await cadeia_praia.ainvoke({"query": estado["query"]})
    return {
        **estado,
        "resposta": resposta
    }

Nó de montanha

In [11]:
async def no_montanha(estado: Estado) -> Estado:
    resposta = await cadeia_montanha.ainvoke({"query": estado["query"]})
    return {
        **estado,
        "resposta": resposta
    }

Função de decisão

In [13]:
def escolher_no(estado: Estado) -> Literal["praia", "montanha"]:
    if estado["destino"] == "praia":
        return "praia"
    return "montanha"

Montagem do grafo

In [14]:
grafo = StateGraph(Estado)

grafo.add_node("roteador", no_roteador)
grafo.add_node("praia", no_praia)
grafo.add_node("montanha", no_montanha)

grafo.add_edge(START, "roteador")

grafo.add_conditional_edges(
    "roteador",
    escolher_no,
    {
        "praia": "praia",
        "montanha": "montanha"
    }
)

grafo.add_edge("praia", END)
grafo.add_edge("montanha", END)

app = grafo.compile()

Execução assíncrona

In [15]:
async def executar_fluxo(pergunta: str):
    estado_inicial = {
        "query": pergunta,
        "destino": "",
        "resposta": ""
    }

    resultado = await app.ainvoke(estado_inicial)

    print("Pergunta:", resultado["query"])
    print("Destino escolhido:", resultado["destino"])
    print("Resposta final:", resultado["resposta"])

    return resultado

In [16]:
resultado_praia = await executar_fluxo("Quero viajar para um lugar com praias bonitas e clima animado.")

Pergunta: Quero viajar para um lugar com praias bonitas e clima animado.
Destino escolhido: praia
Resposta final: Excelente! Para combinar praias deslumbrantes com um clima vibrante e animado, tenho algumas sugestões que certamente vão te agradar:

---

### **1. Cancún, México**

*   **Praias Bonitas:** Areias brancas como talco e um mar turquesa inacreditável. As praias da Zona Hoteleira são extensas e deslumbrantes.
*   **Clima Animado:** Famosa por sua vida noturna intensa, com boates renomadas (Coco Bongo, The City), bares e festas que duram até o amanhecer. É um destino muito procurado por jovens e grupos.
*   **Vibe:** Diversão sem limites, resorts all-inclusive e muitas opções de passeios (ruínas maias, cenotes).

---

### **2. Rio de Janeiro, Brasil**

*   **Praias Bonitas:** Cartões-postais como Copacabana e Ipanema, com a beleza da cidade, o Pão de Açúcar e o Cristo Redentor ao fundo. Praias urbanas com paisagens espetaculares.
*   **Clima Animado:** A cidade respira música e

In [17]:
resultado_montanha = await executar_fluxo("Quero uma viagem tranquila para montanha, com frio e natureza.")

ChatGoogleGenerativeAIError: Error calling model 'gemini-2.5-flash' (RESOURCE_EXHAUSTED): 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. \n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 20, model: gemini-2.5-flash\nPlease retry in 55.161575321s.', 'status': 'RESOURCE_EXHAUSTED', 'details': [{'@type': 'type.googleapis.com/google.rpc.Help', 'links': [{'description': 'Learn more about Gemini API quotas', 'url': 'https://ai.google.dev/gemini-api/docs/rate-limits'}]}, {'@type': 'type.googleapis.com/google.rpc.QuotaFailure', 'violations': [{'quotaMetric': 'generativelanguage.googleapis.com/generate_content_free_tier_requests', 'quotaId': 'GenerateRequestsPerDayPerProjectPerModel-FreeT

Versão com debug simples

In [18]:
async def executar_fluxo_debug(pergunta: str):
    estado_inicial = {
        "query": pergunta,
        "destino": "",
        "resposta": ""
    }

    print("=== ESTADO INICIAL ===")
    print(estado_inicial)
    print()

    estado_roteado = await no_roteador(estado_inicial)

    print("=== APÓS ROTEADOR ===")
    print(estado_roteado)
    print()

    proximo = escolher_no(estado_roteado)
    print("=== PRÓXIMO NÓ ===")
    print(proximo)
    print()

    if proximo == "praia":
        estado_final = await no_praia(estado_roteado)
    else:
        estado_final = await no_montanha(estado_roteado)

    print("=== ESTADO FINAL ===")
    print(estado_final)
    return estado_final